<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINTIA/blob/main/notebooks/06_decisao_sem_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 06 · Reconstrução mínima: modelo de decisão só com transformers

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINTIA)

Engenharia reversa da ideia, sem a pilha da Unsloth: em vez de gerar texto, medimos a **probabilidade de cada opção** como continuação da pergunta, normalizamos com softmax e calibramos com *temperature scaling* do núcleo OSINTIA. É *zero-shot* (sem treino) e serve de linha de base para comparar com o notebook 01.

| | |
|---|---|
| **Hardware** | GPU T4 gratuita (Ambiente de execução → Alterar tipo → T4 GPU) |
| **Tempo estimado** | 5 a 10 minutos (T4) · funciona em CPU, mais devagar |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
!pip install -q -U "transformers>=4.45" accelerate

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINTIA.git /content/OSINTIA 2>/dev/null || git -C /content/OSINTIA pull -q
import sys
sys.path.insert(0, "/content/OSINTIA")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

## 1. Um LLM pequeno e comum
Qualquer modelo de chat serve. `Qwen/Qwen2.5-1.5B-Instruct` é leve e entende português.

In [ ]:
import math, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODELO)
lm = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype="auto", device_map="auto")
lm.eval()

## 2. A cabeça de decisão, versão artesanal
Para cada opção, somamos o log da probabilidade dos tokens do nome da opção depois do prompt. As opções viram uma distribuição. Nada é gerado.

In [ ]:
def montar_prompt(state, pergunta):
    linhas = "\n".join(f"- {o}: {d}" for o, d in pergunta["criteria"].items())
    texto = f"{state}\n\nPergunta: {pergunta['instructions']}\nOpções:\n{linhas}\nResponda só com o nome exato de uma opção."
    return tok.apply_chat_template([{"role": "user", "content": texto}], add_generation_prompt=True, tokenize=False)

@torch.no_grad()
def decidir(state, pergunta):
    base = montar_prompt(state, pergunta)
    n_base = len(tok(base, add_special_tokens=False)["input_ids"])
    pontos = {}
    for opcao in pergunta["criteria"]:
        ids = tok(base + opcao, add_special_tokens=False, return_tensors="pt")["input_ids"].to(lm.device)
        logp = torch.log_softmax(lm(ids).logits[0, :-1].float(), dim=-1)
        alvo = ids[0, 1:]
        pontos[opcao] = logp[n_base - 1:, :].gather(1, alvo[n_base - 1:, None]).sum().item()
    maior = max(pontos.values())
    expo = {o: math.exp(v - maior) for o, v in pontos.items()}
    soma = sum(expo.values())
    return {o: v / soma for o, v in expo.items()}

print(decidir("Perfil analisado: @bancoaur0ra\nBio: Mande seu CPF no direct para estorno.", esquemas.TRIAGEM_PERFIL["perfil"]))

## 3. Calibrar e auditar
Mesma divisão do notebook 01: auditoria humana separada; calibração em outra parte.

In [ ]:
linhas = dados.carregar_jsonl("/content/OSINTIA/dados/demo/triagem_perfis.jsonl")
resto, aud = dados.dividir_holdout(linhas, fracao=0.2, semente=3407)
calib = resto[:80]
pergunta = esquemas.TRIAGEM_PERFIL["perfil"]

p_calib = [decidir(l["state"], pergunta) for l in calib]
T = calibracao.ajustar_temperatura(p_calib, dados.rotulos(calib, "perfil"))
print(f"temperatura = {T:.2f}")

p_aud = [calibracao.aplicar_temperatura(decidir(l["state"], pergunta), T) for l in aud]
gold_aud = dados.rotulos(aud, "perfil")
humano_aud = [l["rotulo_humano"]["perfil"] for l in aud]
prev = [max(p, key=p.get) for p in p_aud]
print("vs gold:  ", calibracao.relatorio(p_aud, gold_aud))
print("vs humano:", calibracao.relatorio(p_aud, humano_aud))
print(auditoria.lacuna_humana(prev, gold_aud, humano_aud))

### O que comparar com o notebook 01
- O zero-shot costuma errar mais, mas **não herda o vício do professor**, porque nunca viu o gold. Se ele concordar mais com o humano do que o modelo treinado, o problema está no rótulo, não no modelo.
- Cada opção custa uma passada do modelo; a cabeça da Unsloth pontua todas as perguntas numa passada só. É aí que mora o ganho de velocidade.

In [ ]:
import os
os.makedirs("/content/saidas", exist_ok=True)
dados.salvar_jsonl([{"id": l["id"], "probabilidades": p} for l, p in zip(aud, p_aud)], "/content/saidas/zero_shot.jsonl")
custodia.registrar("/content/saidas/manifesto.jsonl", "decisao_zero_shot", "/content/saidas/zero_shot.jsonl", {"modelo": MODELO, "T": round(T, 3)})

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*